In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast, split, lit

spark = SparkSession.builder \
  .appName("IcebergTableManagement") \
  .config("spark.executor.memory", "4g") \
  .config("spark.driver.memory", "4g") \
  .config("spark.sql.shuffle.partitions", "200") \
  .config("spark.sql.files.maxPartitionBytes", "134217728") \
  .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
  .config("spark.dynamicAllocation.enabled", "true") \
  .config("spark.dynamicAllocation.minExecutors", "1") \
  .config("spark.dynamicAllocation.maxExecutors", "50") \
  .getOrCreate()


24/12/14 21:25:20 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
spark.sql('show databases').show()

+---------+
|namespace|
+---------+
+---------+



In [4]:
spark.sql('CREATE DATABASE hw3')

DataFrame[]

In [5]:
spark.sql('show databases').show()

+---------+
|namespace|
+---------+
|      hw3|
+---------+



In [9]:
matches = spark.read.option("header", "true") \
                        .option("inferSchema", "true") \
                        .csv("/home/iceberg/data/matches.csv") \

matchDetails = spark.read.option("header", "true") \
                        .option("inferSchema", "true") \
                        .csv("/home/iceberg/data/match_details.csv") \

medalsMatchesPlayers = spark.read.option("header", "true") \
                        .option("inferSchema", "true") \
                        .csv("/home/iceberg/data/medals_matches_players.csv") \

maps =  spark.read.option("header", "true") \
                        .option("inferSchema", "true") \
                        .csv("/home/iceberg/data/maps.csv") \

medals =  spark.read.option("header", "true") \
                        .option("inferSchema", "true") \
                        .csv("/home/iceberg/data/medals.csv")

In [12]:
matchDetails.columns

['match_id',
 'player_gamertag',
 'previous_spartan_rank',
 'spartan_rank',
 'previous_total_xp',
 'total_xp',
 'previous_csr_tier',
 'previous_csr_designation',
 'previous_csr',
 'previous_csr_percent_to_next_tier',
 'previous_csr_rank',
 'current_csr_tier',
 'current_csr_designation',
 'current_csr',
 'current_csr_percent_to_next_tier',
 'current_csr_rank',
 'player_rank_on_team',
 'player_finished',
 'player_average_life',
 'player_total_kills',
 'player_total_headshots',
 'player_total_weapon_damage',
 'player_total_shots_landed',
 'player_total_melee_kills',
 'player_total_melee_damage',
 'player_total_assassinations',
 'player_total_ground_pound_kills',
 'player_total_shoulder_bash_kills',
 'player_total_grenade_damage',
 'player_total_power_weapon_damage',
 'player_total_power_weapon_grabs',
 'player_total_deaths',
 'player_total_assists',
 'player_total_grenade_kills',
 'did_win',
 'team_id']

In [13]:
bucketedMatchDetails = """
CREATE TABLE IF NOT EXISTS hw3.match_details_bucketed (
    match_id STRING,
    player_gamertag STRING,
    player_total_kills INTEGER,
    player_total_deaths INTEGER
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(bucketedMatchDetails)

DataFrame[]

In [15]:
matchDetails.select("match_id", "player_gamertag", "player_total_kills", "player_total_deaths") \
    .write.mode("append") \
    .bucketBy(16, "match_id").saveAsTable("hw3.match_details_bucketed")

In [16]:
medalsMatchesPlayers.columns

['match_id', 'player_gamertag', 'medal_id', 'count']

In [17]:
bucketedMedalsMatchesPlayersDetails = """
CREATE TABLE IF NOT EXISTS hw3.medals_matches_players (
    match_id STRING,
    player_gamertag STRING,
    medal_id STRING,
    count INTEGER
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(bucketedMedalsMatchesPlayersDetails)

DataFrame[]

In [18]:
medalsMatchesPlayers.select("match_id", "player_gamertag", "medal_id", "count") \
    .write.mode("append") \
    .bucketBy(16, "match_id").saveAsTable("hw3.medals_matches_players")

In [19]:
spark.sql('show tables in hw3').show()

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|      hw3|match_details_buc...|      false|
|      hw3|medals_matches_pl...|      false|
+---------+--------------------+-----------+



In [20]:
distinctDates = matches.select("completion_date").distinct().collect()

In [21]:
matches.columns

['match_id',
 'mapid',
 'is_team_game',
 'playlist_id',
 'game_variant_id',
 'is_match_over',
 'completion_date',
 'match_duration',
 'game_mode',
 'map_variant_id']

In [22]:
matchesBucketedDDL = """
CREATE TABLE IF NOT EXISTS hw3.matches_bucketed (
    match_id STRING,
    mapid STRING,
    is_team_game BOOLEAN,
    playlist_id STRING,
    completion_date TIMESTAMP
)
USING iceberg
PARTITIONED BY (completion_date, bucket(16, match_id))
"""
spark.sql(matchesBucketedDDL)

DataFrame[]

In [23]:
distinctDates

[Row(completion_date=datetime.datetime(2016, 3, 13, 0, 0)),
 Row(completion_date=datetime.datetime(2016, 3, 11, 0, 0)),
 Row(completion_date=datetime.datetime(2016, 3, 10, 0, 0)),
 Row(completion_date=datetime.datetime(2016, 1, 30, 0, 0)),
 Row(completion_date=datetime.datetime(2016, 3, 27, 0, 0)),
 Row(completion_date=datetime.datetime(2016, 4, 10, 0, 0)),
 Row(completion_date=datetime.datetime(2016, 1, 18, 0, 0)),
 Row(completion_date=datetime.datetime(2016, 2, 1, 0, 0)),
 Row(completion_date=datetime.datetime(2015, 12, 14, 0, 0)),
 Row(completion_date=datetime.datetime(2016, 2, 3, 0, 0)),
 Row(completion_date=datetime.datetime(2016, 4, 30, 0, 0)),
 Row(completion_date=datetime.datetime(2016, 3, 5, 0, 0)),
 Row(completion_date=datetime.datetime(2016, 4, 15, 0, 0)),
 Row(completion_date=datetime.datetime(2016, 5, 21, 0, 0)),
 Row(completion_date=datetime.datetime(2015, 10, 31, 0, 0)),
 Row(completion_date=datetime.datetime(2016, 1, 22, 0, 0)),
 Row(completion_date=datetime.datetime(20

In [27]:
distinctDates[0]['completion_date'] == (2016, 3, 13, 0, 0)

False

In [28]:
from pyspark.sql.functions import to_timestamp

In [32]:
to_timestamp(("2016, 3, 13, 0, 0"))

Column<'to_timestamp(2016, 3, 13, 0, 0)'>

In [39]:
from pyspark.storagelevel import StorageLevel

In [42]:
for row in distinctDates:
    date = row['completion_date']
    filteredMatches = matches.filter(matches.completion_date == date)
    optimizedMatches = filteredMatches \
        .select("match_id", "mapid", "is_team_game", "playlist_id", "completion_date") \
        .repartition(16, "match_id") \
        .persist(StorageLevel.MEMORY_AND_DISK)

    optimizedMatches.write \
    .mode("append") \
    .bucketBy(16, "match_id") \
    .partitionBy("completion_date") \
    .saveAsTable("hw3.matches_bucketed")

In [43]:
matches_result = spark.sql("SELECT * FROM hw3.matches_bucketed")
matches_result.show()

+--------------------+--------------------+------------+--------------------+-------------------+
|            match_id|               mapid|is_team_game|         playlist_id|    completion_date|
+--------------------+--------------------+------------+--------------------+-------------------+
|14b60c7a-afb1-403...|c7edbf0f-f206-11e...|        NULL|f72e0ef0-7c4a-430...|2015-12-24 00:00:00|
|552b4a88-73b4-415...|cae999f0-f206-11e...|        true|0e39ead4-383b-445...|2015-12-24 00:00:00|
|acd50816-c02d-421...|cae999f0-f206-11e...|        true|0e39ead4-383b-445...|2015-12-24 00:00:00|
|0eedeb83-f9da-46f...|c7edbf0f-f206-11e...|        NULL|f72e0ef0-7c4a-430...|2015-12-24 00:00:00|
|27d403ba-6a5b-468...|c7edbf0f-f206-11e...|        NULL|f72e0ef0-7c4a-430...|2015-12-24 00:00:00|
|5a6efada-fc0d-41a...|c96ff240-f206-11e...|        true|b50c4dc2-6c86-4d7...|2015-12-24 00:00:00|
|ee17e9cb-0aaf-4c2...|c7b7baf0-f206-11e...|        true|780cc101-005c-4fc...|2015-12-24 00:00:00|
|9b58a9a9-9bf1-4e9..

In [46]:
spark.sql('SELECT * from hw3.match_details_bucketed mdb JOIN hw3.matches_bucketed mb ON mdb.match_id = mb.match_id').explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [match_id#46653], [match_id#46657], Inner
   :- Sort [match_id#46653 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(match_id#46653, 200), ENSURE_REQUIREMENTS, [plan_id=16251]
   :     +- BatchScan demo.hw3.match_details_bucketed[match_id#46653, player_gamertag#46654, player_total_kills#46655, player_total_deaths#46656] demo.hw3.match_details_bucketed (branch=null) [filters=match_id IS NOT NULL, groupedBy=] RuntimeFilters: []
   +- Sort [match_id#46657 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(match_id#46657, 200), ENSURE_REQUIREMENTS, [plan_id=16252]
         +- BatchScan demo.hw3.matches_bucketed[match_id#46657, mapid#46658, is_team_game#46659, playlist_id#46660, completion_date#46661] demo.hw3.matches_bucketed (branch=null) [filters=match_id IS NOT NULL, groupedBy=] RuntimeFilters: []




In [48]:
matches.createOrReplaceTempView("matchesView")

In [49]:
matchDetails.createOrReplaceTempView("matchDetailsView")

In [51]:
spark.sql('SELECT * from matchDetailsView mdv JOIN matchesView mv ON mdv.match_id = mv.match_id').explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [match_id#72], [match_id#35], Inner
   :- Sort [match_id#72 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(match_id#72, 200), ENSURE_REQUIREMENTS, [plan_id=16290]
   :     +- Filter isnotnull(match_id#72)
   :        +- FileScan csv [match_id#72,player_gamertag#73,previous_spartan_rank#74,spartan_rank#75,previous_total_xp#76,total_xp#77,previous_csr_tier#78,previous_csr_designation#79,previous_csr#80,previous_csr_percent_to_next_tier#81,previous_csr_rank#82,current_csr_tier#83,current_csr_designation#84,current_csr#85,current_csr_percent_to_next_tier#86,current_csr_rank#87,player_rank_on_team#88,player_finished#89,player_average_life#90,player_total_kills#91,player_total_headshots#92,player_total_weapon_damage#93,player_total_shots_landed#94,player_total_melee_kills#95,... 12 more fields] Batched: false, DataFilters: [isnotnull(match_id#72)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/h

In [64]:
createBucketJoinedDDL = """
CREATE TABLE IF NOT EXISTS hw3.final_matches_joined
USING iceberg
PARTITIONED BY (bucket(16, match_id))
AS
(
    SELECT mdb.match_id, mdb.player_gamertag, mdb.player_total_kills, mdb.player_total_deaths,
        mb.mapid, mb.is_team_game, mb.playlist_id, mb.completion_date,
        mmp.medal_id, mmp.count
    FROM ((hw3.match_details_bucketed mdb
    INNER JOIN hw3.matches_bucketed mb ON mdb.match_id = mb.match_id)
    INNER JOIN hw3.medals_matches_players mmp ON mdb.match_id = mmp.match_id)
)
"""

In [53]:
select = """
SELECT * FROM ((hw3.match_details_bucketed mdb
    INNER JOIN hw3.matches_bucketed mb ON mdb.match_id = mb.match_id)
    INNER JOIN hw3.medals_matches_players mmp ON mdb.match_id = mmp.match_id)
    """

In [54]:
spark.sql(select).show()

+--------------------+---------------+------------------+-------------------+--------------------+--------------------+------------+--------------------+-------------------+--------------------+---------------+----------+-----+
|            match_id|player_gamertag|player_total_kills|player_total_deaths|            match_id|               mapid|is_team_game|         playlist_id|    completion_date|            match_id|player_gamertag|  medal_id|count|
+--------------------+---------------+------------------+-------------------+--------------------+--------------------+------------+--------------------+-------------------+--------------------+---------------+----------+-----+
|00169217-cca6-4b4...|  King Terror V|                14|                  7|00169217-cca6-4b4...|cc040aa1-f206-11e...|        true|2323b76a-db98-4e0...|2016-03-13 00:00:00|00169217-cca6-4b4...|  King Terror V|3261908037|   11|
|00169217-cca6-4b4...|  King Terror V|                14|                  7|00169217-cc

In [65]:
spark.sql(createBucketJoinedDDL)

DataFrame[]

In [66]:
spark.sql('select * from hw3.final_matches_joined').show()

+--------------------+---------------+------------------+-------------------+--------------------+------------+--------------------+-------------------+----------+-----+
|            match_id|player_gamertag|player_total_kills|player_total_deaths|               mapid|is_team_game|         playlist_id|    completion_date|  medal_id|count|
+--------------------+---------------+------------------+-------------------+--------------------+------------+--------------------+-------------------+----------+-----+
|0377e616-bf8b-4a4...|     Snipe Envy|                14|                  8|cdee4e70-f206-11e...|        true|bc0f8ad6-31e6-4a1...|2016-04-23 00:00:00|2078758684|    2|
|0377e616-bf8b-4a4...|     Snipe Envy|                14|                  8|cdee4e70-f206-11e...|        true|bc0f8ad6-31e6-4a1...|2016-04-23 00:00:00| 848240062|    1|
|0377e616-bf8b-4a4...|     Snipe Envy|                14|                  8|cdee4e70-f206-11e...|        true|bc0f8ad6-31e6-4a1...|2016-04-23 00:00:0

In [67]:
spark.sql('select count(*) from hw3.final_matches_joined').show()

+--------+
|count(1)|
+--------+
| 6885858|
+--------+

